In [5]:
import torch
from pathlib import Path
from torchvision import datasets
from torch.utils.data import random_split

# Use the existing dataset directly
DATASET_DIR = Path("../dataset/suitability")

dataset = datasets.ImageFolder(DATASET_DIR)

print("Total images:", len(dataset))
print("Classes:", dataset.class_to_idx)

# 70% train, 15% validation, 15% test
train_size = int(0.70 * len(dataset))
val_size = int(0.15 * len(dataset))
test_size = len(dataset) - train_size - val_size

generator = torch.Generator().manual_seed(42)

train_dataset, val_dataset, test_dataset = random_split(
    dataset,
    [train_size, val_size, test_size],
    generator=generator
)

print()
print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))

Total images: 14202
Classes: {'suitable': 0, 'unsuitable': 1}

Train: 9941
Validation: 2130
Test: 2131


In [6]:
import numpy as np
from sklearn.model_selection import train_test_split
from torch.utils.data import Subset

# All labels from ImageFolder
targets = np.array(dataset.targets)
indices = np.arange(len(dataset))

# 70% train, 30% temporary
train_idx, temp_idx = train_test_split(
    indices,
    test_size=0.30,
    stratify=targets,
    random_state=42
)

# Split remaining 30% into 15% validation and 15% test
temp_targets = targets[temp_idx]

val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.50,
    stratify=temp_targets,
    random_state=42
)

train_dataset = Subset(dataset, train_idx)
val_dataset = Subset(dataset, val_idx)
test_dataset = Subset(dataset, test_idx)

print("Total:", len(dataset))
print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))

def count_classes(indices):
    values = targets[indices]
    return {
        "suitable": int(np.sum(values == 0)),
        "unsuitable": int(np.sum(values == 1))
    }

print("\nClass distribution:")
print("Train:", count_classes(train_idx))
print("Validation:", count_classes(val_idx))
print("Test:", count_classes(test_idx))

Total: 14202
Train: 9941
Validation: 2130
Test: 2131

Class distribution:
Train: {'suitable': 8681, 'unsuitable': 1260}
Validation: {'suitable': 1860, 'unsuitable': 270}
Test: {'suitable': 1861, 'unsuitable': 270}


In [7]:
from torchvision import transforms
from torch.utils.data import DataLoader

IMAGE_SIZE = 224
BATCH_SIZE = 16

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_test_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# Create separate ImageFolder datasets so each split can use
# the appropriate transforms without copying any images.
full_dataset_train = datasets.ImageFolder(DATASET_DIR, transform=train_transform)
full_dataset_eval = datasets.ImageFolder(DATASET_DIR, transform=val_test_transform)

train_dataset = Subset(full_dataset_train, train_idx)
val_dataset = Subset(full_dataset_eval, val_idx)
test_dataset = Subset(full_dataset_eval, test_idx)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("DataLoaders ready!")
print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

DataLoaders ready!
Train batches: 622
Validation batches: 134
Test batches: 134


In [8]:
import torch
import torch.nn as nn
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# EfficientNet-B0 pretrained on ImageNet
weights = EfficientNet_B0_Weights.DEFAULT
model = efficientnet_b0(weights=weights)

# Replace final layer for binary classification
model.classifier[1] = nn.Linear(
    model.classifier[1].in_features,
    1
)

model = model.to(device)

# Handle class imbalance
# class 0 = suitable
# class 1 = unsuitable
pos_weight = torch.tensor(
    [8681 / 1260],
    device=device,
    dtype=torch.float32
)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

print("Model:", "EfficientNet-B0")
print("Positive-class weight:", pos_weight.item())
print("Model ready!")

Device: cuda
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /home/dhruv/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|█████████████████████████████████████████████████████████████████████████████| 20.5M/20.5M [00:06<00:00, 3.11MB/s]


Model: EfficientNet-B0
Positive-class weight: 6.889682769775391
Model ready!


In [9]:
from tqdm.auto import tqdm

NUM_EPOCHS = 5

history = {
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": []
}

best_val_acc = 0.0

for epoch in range(NUM_EPOCHS):

    # -------------------------
    # Training
    # -------------------------
    model.train()

    train_loss = 0.0
    train_correct = 0
    train_total = 0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} - Train")

    for images, labels in pbar:
        images = images.to(device)
        labels = labels.float().unsqueeze(1).to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item() * images.size(0)

        predictions = (torch.sigmoid(outputs) >= 0.5).float()

        train_correct += (predictions == labels).sum().item()
        train_total += labels.size(0)

        pbar.set_postfix(loss=f"{loss.item():.4f}")

    train_loss /= train_total
    train_acc = train_correct / train_total

    # -------------------------
    # Validation
    # -------------------------
    model.eval()

    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():

        pbar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} - Val")

        for images, labels in pbar:
            images = images.to(device)
            labels = labels.float().unsqueeze(1).to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * images.size(0)

            predictions = (torch.sigmoid(outputs) >= 0.5).float()

            val_correct += (predictions == labels).sum().item()
            val_total += labels.size(0)

    val_loss /= val_total
    val_acc = val_correct / val_total

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    print()
    print(f"Epoch {epoch+1}/{NUM_EPOCHS}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Train Acc:  {train_acc:.4f}")
    print(f"Val Loss:   {val_loss:.4f}")
    print(f"Val Acc:    {val_acc:.4f}")

    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc

        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "model_name": "efficientnet_b0",
                "class_names": ["suitable", "unsuitable"],
                "threshold": 0.5,
                "image_size": 224
            },
            "../models/lesion_suitability_efficientnet_b0.pth"
        )

        print("✓ Best model saved")

print()
print("Training complete!")
print(f"Best validation accuracy: {best_val_acc:.4f}")

Epoch 1/5 - Train:   0%|          | 0/622 [00:00<?, ?it/s]

Epoch 1/5 - Val:   0%|          | 0/134 [00:00<?, ?it/s]


Epoch 1/5
Train Loss: 0.5095
Train Acc:  0.8455
Val Loss:   0.3860
Val Acc:    0.8812
✓ Best model saved


Epoch 2/5 - Train:   0%|          | 0/622 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x72f395317e20>
Traceback (most recent call last):
  File "/home/dhruv/melanoma-project/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py", line 1704, in __del__
    self._shutdown_workers()
  File "/home/dhruv/melanoma-project/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py", line 1673, in _shutdown_workers
    if w.is_alive():
   Exception ignored in:  <function _MultiProcessingDataLoaderIter.__del__ at 0x72f395317e20> 
 Traceback (most recent call last):
   File "/home/dhruv/melanoma-project/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py", line 1704, in __del__
^    self._shutdown_workers()^^
^^  File "/home/dhruv/melanoma-project/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py", line 1673, in _shutdown_workers
^^    ^if w.is_alive():^^
^ ^ 
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
      assert self._pa

...

Epoch 4/5 - Val:   0%|          | 0/134 [00:00<?, ?it/s]


Epoch 4/5
Train Loss: 0.4027
Train Acc:  0.8683
Val Loss:   0.3776
Val Acc:    0.8798


Epoch 5/5 - Train:   0%|          | 0/622 [00:00<?, ?it/s]

Epoch 5/5 - Val:   0%|          | 0/134 [00:00<?, ?it/s]


Epoch 5/5
Train Loss: 0.3943
Train Acc:  0.8711
Val Loss:   0.3777
Val Acc:    0.8812

Training complete!
Best validation accuracy: 0.8812


In [10]:
import torch
from tqdm.auto import tqdm

model.eval()

correct = 0
total = 0

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Testing"):
        images = images.to(device)
        labels = labels.float().unsqueeze(1).to(device)

        outputs = model(images)
        probabilities = torch.sigmoid(outputs)
        predictions = (probabilities >= 0.5).float()

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

test_accuracy = correct / total

print(f"Test Accuracy: {test_accuracy:.4f}")
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")

Testing:   0%|          | 0/134 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x72f395317e20>
Traceback (most recent call last):
  File "/home/dhruv/melanoma-project/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py", line 1704, in __del__
    self._shutdown_workers()
  File "/home/dhruv/melanoma-project/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py", line 1673, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x72f395317e20>
Traceback (most recent call last):
  File "/home/dhruv/melanoma-project/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py", line 1704, in __del__
    self._shutdown_workers()
  File "/home/dh

Test Accuracy: 0.8667
Test Accuracy: 86.67%


In [11]:
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    precision_score,
    recall_score,
    f1_score
)
import numpy as np
import torch

model.eval()

all_labels = []
all_predictions = []
all_probabilities = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)

        outputs = model(images)
        probabilities = torch.sigmoid(outputs).squeeze(1)

        predictions = (probabilities >= 0.5).long()

        all_labels.extend(labels.numpy())
        all_predictions.extend(predictions.cpu().numpy())
        all_probabilities.extend(probabilities.cpu().numpy())

all_labels = np.array(all_labels)
all_predictions = np.array(all_predictions)

print("Confusion Matrix:")
print(confusion_matrix(all_labels, all_predictions))

print("\nClassification Report:")
print(
    classification_report(
        all_labels,
        all_predictions,
        target_names=["suitable", "unsuitable"],
        digits=4
    )
)

print("\nSummary:")
print(f"Precision: {precision_score(all_labels, all_predictions):.4f}")
print(f"Recall:    {recall_score(all_labels, all_predictions):.4f}")
print(f"F1 Score:  {f1_score(all_labels, all_predictions):.4f}")

Confusion Matrix:
[[1582  279]
 [   5  265]]

Classification Report:
              precision    recall  f1-score   support

    suitable     0.9968    0.8501    0.9176      1861
  unsuitable     0.4871    0.9815    0.6511       270

    accuracy                         0.8667      2131
   macro avg     0.7420    0.9158    0.7844      2131
weighted avg     0.9323    0.8667    0.8839      2131


Summary:
Precision: 0.4871
Recall:    0.9815
F1 Score:  0.6511


In [12]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score
import numpy as np

thresholds = [0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70]

print("Threshold Analysis")
print("-" * 75)
print(f"{'Threshold':<12}{'Suitable Recall':<18}{'Unsuitable Recall':<20}{'Accuracy':<12}")

for threshold in thresholds:
    predictions = (np.array(all_probabilities) >= threshold).astype(int)

    cm = confusion_matrix(all_labels, predictions)

    suitable_recall = recall_score(
        all_labels,
        predictions,
        pos_label=0
    )

    unsuitable_recall = recall_score(
        all_labels,
        predictions,
        pos_label=1
    )

    accuracy = np.mean(predictions == all_labels)

    print(
        f"{threshold:<12.2f}"
        f"{suitable_recall:<18.4f}"
        f"{unsuitable_recall:<20.4f}"
        f"{accuracy:<12.4f}"
    )

Threshold Analysis
---------------------------------------------------------------------------
Threshold   Suitable Recall   Unsuitable Recall   Accuracy    
0.30        0.8501            0.9889              0.8677      
0.35        0.8501            0.9852              0.8672      
0.40        0.8501            0.9852              0.8672      
0.45        0.8501            0.9815              0.8667      
0.50        0.8501            0.9815              0.8667      
0.55        0.8501            0.9815              0.8667      
0.60        0.8501            0.9815              0.8667      
0.65        0.8501            0.9778              0.8663      
0.70        0.8501            0.9704              0.8653      


In [13]:
thresholds = [
    0.01, 0.02, 0.03, 0.05,
    0.08, 0.10, 0.15, 0.20,
    0.25, 0.30
]

print("Lower Threshold Analysis")
print("-" * 75)
print(f"{'Threshold':<12}{'Suitable Recall':<18}{'Unsuitable Recall':<20}{'Accuracy':<12}")

for threshold in thresholds:
    predictions = (np.array(all_probabilities) >= threshold).astype(int)

    suitable_recall = recall_score(
        all_labels,
        predictions,
        pos_label=0
    )

    unsuitable_recall = recall_score(
        all_labels,
        predictions,
        pos_label=1
    )

    accuracy = np.mean(predictions == all_labels)

    print(
        f"{threshold:<12.2f}"
        f"{suitable_recall:<18.4f}"
        f"{unsuitable_recall:<20.4f}"
        f"{accuracy:<12.4f}"
    )

Lower Threshold Analysis
---------------------------------------------------------------------------
Threshold   Suitable Recall   Unsuitable Recall   Accuracy    
0.01        0.8216            1.0000              0.8442      
0.02        0.8356            0.9926              0.8555      
0.03        0.8409            0.9926              0.8602      
0.05        0.8458            0.9926              0.8644      
0.08        0.8474            0.9926              0.8658      
0.10        0.8479            0.9926              0.8663      
0.15        0.8490            0.9926              0.8672      
0.20        0.8495            0.9926              0.8677      
0.25        0.8495            0.9889              0.8672      
0.30        0.8501            0.9889              0.8677      


In [14]:
import numpy as np

probs = np.array(all_probabilities)
labels = np.array(all_labels)

suitable_probs = probs[labels == 0]
unsuitable_probs = probs[labels == 1]

print("SUITABLE probability distribution")
print("Min:    ", suitable_probs.min())
print("25%:    ", np.percentile(suitable_probs, 25))
print("Median: ", np.median(suitable_probs))
print("75%:    ", np.percentile(suitable_probs, 75))
print("Max:    ", suitable_probs.max())

print("\nUNSUITABLE probability distribution")
print("Min:    ", unsuitable_probs.min())
print("25%:    ", np.percentile(unsuitable_probs, 25))
print("Median: ", np.median(unsuitable_probs))
print("75%:    ", np.percentile(unsuitable_probs, 75))
print("Max:    ", unsuitable_probs.max())

SUITABLE probability distribution
Min:     7.569444e-09
25%:     5.551788e-05
Median:  0.0003175657
75%:     0.0020752244
Max:     0.9316511

UNSUITABLE probability distribution
Min:     0.017056506
25%:     0.86951953
Median:  0.8809446
75%:     0.8918742
Max:     0.92489797


In [15]:
# Replace the heavily weighted loss with standard binary cross entropy

criterion = nn.BCEWithLogitsLoss()

print("Loss function changed to standard BCE")

Loss function changed to standard BCE


In [16]:

from tqdm.auto import tqdm

NUM_EPOCHS = 5

history = {
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": []
}

best_val_acc = 0.0

for epoch in range(NUM_EPOCHS):

    # =========================
    # Training
    # =========================
    model.train()

    train_loss = 0.0
    train_correct = 0
    train_total = 0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} - Train")

    for images, labels in pbar:
        images = images.to(device)
        labels = labels.float().unsqueeze(1).to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item() * images.size(0)

        predictions = (torch.sigmoid(outputs) >= 0.5).float()

        train_correct += (predictions == labels).sum().item()
        train_total += labels.size(0)

        pbar.set_postfix(loss=f"{loss.item():.4f}")

    train_loss /= train_total
    train_acc = train_correct / train_total

    # =========================
    # Validation
    # =========================
    model.eval()

    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():

        pbar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} - Val")

        for images, labels in pbar:
            images = images.to(device)
            labels = labels.float().unsqueeze(1).to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * images.size(0)

            predictions = (torch.sigmoid(outputs) >= 0.5).float()

            val_correct += (predictions == labels).sum().item()
            val_total += labels.size(0)

    val_loss /= val_total
    val_acc = val_correct / val_total

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    print()
    print(f"Epoch {epoch+1}/{NUM_EPOCHS}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Train Acc:  {train_acc:.4f}")
    print(f"Val Loss:   {val_loss:.4f}")
    print(f"Val Acc:    {val_acc:.4f}")

    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc

        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "model_name": "efficientnet_b0",
                "class_names": ["suitable", "unsuitable"],
                "threshold": 0.5,
                "image_size": 224
            },
            "../models/lesion_suitability_efficientnet_b0.pth"
        )

        print("✓ Best model saved")

print()
print("Training complete!")
print(f"Best validation accuracy: {best_val_acc:.4f}")

Epoch 1/5 - Train:   0%|          | 0/622 [00:00<?, ?it/s]

Epoch 1/5 - Val:   0%|          | 0/134 [00:00<?, ?it/s]


Epoch 1/5
Train Loss: 0.1844
Train Acc:  0.8755
Val Loss:   0.1780
Val Acc:    0.8474
✓ Best model saved


Epoch 2/5 - Train:   0%|          | 0/622 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x72f395317e20>
Traceback (most recent call last):
  File "/home/dhruv/melanoma-project/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py", line 1704, in __del__
    self._shutdown_workers()
  File "/home/dhruv/melanoma-project/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py", line 1673, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
   Exception ignored in:  <function _MultiProcessingDataLoaderIter.__del__ at 0x72f395317e20> 
 Traceback (most recent call last):
   File "/home/dhruv/melanoma-project/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py", line 1704, in __del__
       self._shutdown_workers() 
^  File "/home/dhruv/melanoma-project/.venv/lib/python3.12/site-packages/torch/utils/data/data

...

Epoch 5/5 - Train:   0%|          | 0/622 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x72f395317e20>
Traceback (most recent call last):
  File "/home/dhruv/melanoma-project/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py", line 1704, in __del__
    self._shutdown_workers()
  File "/home/dhruv/melanoma-project/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py", line 1673, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x72f395317e20>
Traceback (most recent call last):
  File "/home/dhruv/melanoma-project/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py", line 1704, in __del__
    self._shutdown_workers()
  File "/home/dh

Epoch 5/5 - Val:   0%|          | 0/134 [00:00<?, ?it/s]

Exception ignored in: Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x72f395317e20><function _MultiProcessingDataLoaderIter.__del__ at 0x72f395317e20>

Traceback (most recent call last):
Traceback (most recent call last):
  File "/home/dhruv/melanoma-project/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py", line 1704, in __del__
  File "/home/dhruv/melanoma-project/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py", line 1704, in __del__
        self._shutdown_workers()self._shutdown_workers()

  File "/home/dhruv/melanoma-project/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py", line 1673, in _shutdown_workers
  File "/home/dhruv/melanoma-project/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py", line 1673, in _shutdown_workers
        if w.is_alive():if w.is_alive():

              ^^^^^^^^^^^^^^^^^^^^^Exception ignored in: ^^<function _MultiProcessingDataLoaderIter.__del__ at 0x72f395317


Epoch 5/5
Train Loss: 0.1777
Train Acc:  0.8778
Val Loss:   0.1849
Val Acc:    0.8657
✓ Best model saved

Training complete!
Best validation accuracy: 0.8657


In [18]:
# Recreate DataLoaders with single-process loading
# This avoids the Jupyter/WSL multiprocessing warning.

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

print("DataLoaders recreated successfully.")
print("Train:", len(train_loader))
print("Validation:", len(val_loader))
print("Test:", len(test_loader))

DataLoaders recreated successfully.
Train: 622
Validation: 134
Test: 134


In [19]:
import torch
import torch.nn as nn
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from tqdm.auto import tqdm

# Create a fresh EfficientNet-B0
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, 1)
model = model.to(device)

# Weighted BCE used in the better previous run
pos_weight = torch.tensor(
    [8681 / 1260],
    device=device,
    dtype=torch.float32
)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

print("Fresh weighted model ready.")
print("Device:", device)
print("Positive-class weight:", pos_weight.item())

Fresh weighted model ready.
Device: cuda
Positive-class weight: 6.889682769775391


In [20]:
from tqdm.auto import tqdm

NUM_EPOCHS = 5

best_val_acc = 0.0

weighted_history = {
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": []
}

for epoch in range(NUM_EPOCHS):

    # =========================
    # Training
    # =========================
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    pbar = tqdm(
        train_loader,
        desc=f"Epoch {epoch+1}/{NUM_EPOCHS} - Train"
    )

    for images, labels in pbar:
        images = images.to(device)
        labels = labels.float().unsqueeze(1).to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

        predictions = (torch.sigmoid(outputs) >= 0.5).float()

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

        pbar.set_postfix(loss=f"{loss.item():.4f}")

    train_loss = running_loss / total
    train_acc = correct / total

    # =========================
    # Validation
    # =========================
    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():

        pbar = tqdm(
            val_loader,
            desc=f"Epoch {epoch+1}/{NUM_EPOCHS} - Val"
        )

        for images, labels in pbar:
            images = images.to(device)
            labels = labels.float().unsqueeze(1).to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)

            predictions = (torch.sigmoid(outputs) >= 0.5).float()

            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    val_loss = running_loss / total
    val_acc = correct / total

    weighted_history["train_loss"].append(train_loss)
    weighted_history["train_acc"].append(train_acc)
    weighted_history["val_loss"].append(val_loss)
    weighted_history["val_acc"].append(val_acc)

    print()
    print(f"Epoch {epoch+1}/{NUM_EPOCHS}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Train Acc:  {train_acc:.4f}")
    print(f"Val Loss:   {val_loss:.4f}")
    print(f"Val Acc:    {val_acc:.4f}")

    # Save best model separately
    if val_acc > best_val_acc:
        best_val_acc = val_acc

        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "model_name": "efficientnet_b0",
                "class_names": ["suitable", "unsuitable"],
                "threshold": 0.5,
                "image_size": 224,
                "pos_weight": pos_weight.item()
            },
            "../models/lesion_suitability_efficientnet_b0_weighted.pth"
        )

        print("✓ Best weighted model saved")

print()
print("Weighted training complete!")
print(f"Best validation accuracy: {best_val_acc:.4f}")

Epoch 1/5 - Train:   0%|          | 0/622 [00:00<?, ?it/s]

Epoch 1/5 - Val:   0%|          | 0/134 [00:00<?, ?it/s]


Epoch 1/5
Train Loss: 0.5119
Train Acc:  0.8493
Val Loss:   0.3899
Val Acc:    0.8775
✓ Best weighted model saved


Epoch 2/5 - Train:   0%|          | 0/622 [00:00<?, ?it/s]

Epoch 2/5 - Val:   0%|          | 0/134 [00:00<?, ?it/s]

...

Epoch 4/5 - Val:   0%|          | 0/134 [00:00<?, ?it/s]


Epoch 4/5
Train Loss: 0.3924
Train Acc:  0.8712
Val Loss:   0.3773
Val Acc:    0.8798
✓ Best weighted model saved


Epoch 5/5 - Train:   0%|          | 0/622 [00:00<?, ?it/s]

Epoch 5/5 - Val:   0%|          | 0/134 [00:00<?, ?it/s]


Epoch 5/5
Train Loss: 0.3985
Train Acc:  0.8690
Val Loss:   0.3803
Val Acc:    0.8812
✓ Best weighted model saved

Weighted training complete!
Best validation accuracy: 0.8812


In [21]:
import torch
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report

# Load the best weighted checkpoint
checkpoint = torch.load(
    "../models/lesion_suitability_efficientnet_b0_weighted.pth",
    map_location=device
)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

all_labels = []
all_predictions = []
all_probabilities = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)

        outputs = model(images)
        probabilities = torch.sigmoid(outputs).squeeze(1)

        predictions = (probabilities >= 0.5).long()

        all_labels.extend(labels.numpy())
        all_predictions.extend(predictions.cpu().numpy())
        all_probabilities.extend(probabilities.cpu().numpy())

all_labels = np.array(all_labels)
all_predictions = np.array(all_predictions)

print("Confusion Matrix:")
print(confusion_matrix(all_labels, all_predictions))

print("\nClassification Report:")
print(
    classification_report(
        all_labels,
        all_predictions,
        target_names=["suitable", "unsuitable"],
        digits=4
    )
)

Confusion Matrix:
[[1582  279]
 [   4  266]]

Classification Report:
              precision    recall  f1-score   support

    suitable     0.9975    0.8501    0.9179      1861
  unsuitable     0.4881    0.9852    0.6528       270

    accuracy                         0.8672      2131
   macro avg     0.7428    0.9176    0.7853      2131
weighted avg     0.9329    0.8672    0.8843      2131

